## station level raw
#### Documentation: https://www.ncei.noaa.gov/pub/data/cdo/documentation/GHCND_documentation.pdf
#### Data Link: https://www.ncei.noaa.gov/pub/data/ghcn/daily/
#### Repository: https://console.cloud.google.com/bigquery?p=bigquery-public-data&d=ghcn_d&page=dataset&project=fluid-vector-398007&ws=!1m4!1m3!3m2!1sbigquery-public-data!2sghcn_d

In [1]:
import pandas as pd

years = range(2021, 2026)

dfs = []

for year in years:
    url = f"https://www.ncei.noaa.gov/pub/data/ghcn/daily/by_year/{year}.csv.gz"
    
    df_year = pd.read_csv(url, compression="gzip", header=None)
    
    df_year.columns = [
        "station_id",
        "date",
        "element",
        "value",
        "m_flag",
        "q_flag",
        "s_flag",
        "obs_time"
    ]
    
    df_year["date"] = pd.to_datetime(df_year["date"], format="%Y%m%d")
    
    dfs.append(df_year)

df = pd.concat(dfs, ignore_index=True)

print(df.head())
print(df.shape)

    station_id       date element  value m_flag q_flag s_flag  obs_time
0  ARM00087582 2021-01-01    TMIN    203    NaN    NaN      S       NaN
1  ARM00087582 2021-01-01    TAVG    253      H    NaN      S       NaN
2  ARM00087593 2021-01-01    TMIN    152    NaN    NaN      S       NaN
3  ARM00087593 2021-01-01    TAVG    276      H    NaN      S       NaN
4  ARM00087596 2021-01-01    TMIN    155    NaN    NaN      S       NaN
(189344721, 8)


In [2]:
#The five core values are:
#PRCP = Precipitation (mm or inches as per user preference, inches to hundredths on Daily Form pdf file)
#SNOW = Snowfall (mm or inches as per user preference, inches to tenths on Daily Form pdf file)
#SNWD = Snow depth (mm or inches as per user preference, inches on Daily Form pdf file)
#TMAX = Maximum temperature (Fahrenheit or Celsius as per user preference, Fahrenheit to tenths on Daily Form pdf file
#TMIN = Minimum temperature (Fahrenheit or Celsius as per user preference, Fahrenheit to tenths on Daily Form pdf file

elements_keep = ["PRCP","SNOW","SNWD","TMAX", "TMIN"]

df = df[df["element"].isin(elements_keep)]

In [3]:
#transpose
df_wide = (
    df.pivot_table(
        index=["station_id", "date"],
        columns="element",
        values="value",
        aggfunc="first"
    )
    .reset_index()
)

print(df_wide.head())

element   station_id       date  PRCP  SNOW  SNWD   TMAX   TMIN
0        ACW00011647 2025-01-01   NaN   NaN   NaN  290.0  230.0
1        ACW00011647 2025-01-02   NaN   NaN   NaN  280.0  230.0
2        ACW00011647 2025-01-03   NaN   NaN   NaN  280.0  220.0
3        ACW00011647 2025-01-04   NaN   NaN   NaN  280.0  220.0
4        ACW00011647 2025-01-05   NaN   NaN   NaN  290.0  240.0


In [4]:
# make sure date is datetime
df_wide["date"] = pd.to_datetime(df_wide["date"])

# set index for resampling
df_weekly = (
    df_wide
    .set_index("date")
    .groupby("station_id")
    .resample("W")
    .mean(numeric_only=True)
    .reset_index()
)

print(df_weekly.head())

element   station_id       date  PRCP  SNOW  SNWD        TMAX        TMIN
0        ACW00011647 2025-01-05   NaN   NaN   NaN  284.000000  228.000000
1        ACW00011647 2025-01-12   NaN   NaN   NaN  287.142857  231.428571
2        ACW00011647 2025-01-19   NaN   NaN   NaN  281.428571  235.714286
3        ACW00011647 2025-01-26   NaN   NaN   NaN  282.857143  242.857143
4        ACW00011647 2025-02-02   NaN   NaN   NaN  284.285714  240.000000


In [5]:
df_weekly.to_csv('climate_station_weekly.csv', index=False)

In [6]:
# station info
path = "/Users/boyapeng/Desktop/Dissertation/Aim2/Data/NOAA-stations.txt"

colspecs = [
    (0,11),   # station_id
    (12,20),  # latitude
    (21,30),  # longitude
    (31,37),  # elevation
    (38,40),  # state
    (41,71),  # name
    (72,75),  # gsn_flag
    (76,79),  # hcn_crn_flag
    (80,85)   # wmo_id
]

columns = [
    "station_id",
    "latitude",
    "longitude",
    "elevation",
    "state",
    "name",
    "gsn_flag",
    "hcn_crn_flag",
    "wmo_id"
]

df_st = pd.read_fwf(path, colspecs=colspecs, names=columns)

print(df_st.head())

    station_id  latitude  longitude  elevation state                   name  \
0  ACW00011604   17.1167   -61.7833       10.1   NaN  ST JOHNS COOLIDGE FLD   
1  ACW00011647   17.1333   -61.7833       19.2   NaN               ST JOHNS   
2  AE000041196   25.3330    55.5170       34.0   NaN    SHARJAH INTER. AIRP   
3  AEM00041194   25.2550    55.3640       10.4   NaN             DUBAI INTL   
4  AEM00041217   24.4330    54.6510       26.8   NaN         ABU DHABI INTL   

  gsn_flag hcn_crn_flag   wmo_id  
0      NaN          NaN      NaN  
1      NaN          NaN      NaN  
2      GSN          NaN  41196.0  
3      NaN          NaN  41194.0  
4      NaN          NaN  41217.0  


In [12]:
import geopandas as gpd
from shapely.geometry import Point

# --------------------------------------------------
# input:
#   df_st with columns:
#   station_id, latitude, longitude, elevation, ...
#
# goal:
#   fill / overwrite state using latitude + longitude
# --------------------------------------------------

# ----- 1) station table -> GeoDataFrame -----
df_st = df_st.copy()

# keep valid coordinates only
df_st = df_st[df_st["latitude"].notna() & df_st["longitude"].notna()].copy()

gdf_st = gpd.GeoDataFrame(
    df_st,
    geometry=gpd.points_from_xy(df_st["longitude"], df_st["latitude"]),
    crs="EPSG:4326"
)

# ----- 2) read US states boundary shapefile -----
# directly read Census shapefile zip from URL
states_url = "https://www2.census.gov/geo/tiger/GENZ2023/shp/cb_2023_us_state_500k.zip"
gdf_states = gpd.read_file(states_url)

# keep only needed columns
gdf_states = gdf_states[["NAME", "STUSPS", "geometry"]].copy()
gdf_states = gdf_states.to_crs(gdf_st.crs)

# ----- 3) spatial join: assign state to each station -----
gdf_st_joined = gpd.sjoin(
    gdf_st,
    gdf_states,
    how="left",
    predicate="within"
)

# ----- 4) fill / overwrite state column -----
# if you want full state name:
gdf_st_joined["state_name"] = gdf_st_joined["NAME"]

# if you want 2-letter USPS code in column "state":
gdf_st_joined["state"] = gdf_st_joined["STUSPS"]

# clean output
df_st_filled = pd.DataFrame(gdf_st_joined.drop(columns=["geometry", "index_right", "NAME", "STUSPS"]))

# ----- 5) save -----
output_path = "/Users/boyapeng/Desktop/Dissertation/Aim2/Data/NOAA_stations_with_state.csv"
df_st_filled.to_csv(output_path, index=False)


In [14]:
print(df_st_filled.loc[df_st_filled["state"].notna(), ["station_id", "latitude", "longitude", "state"]].head(20))
print(df_st_filled["state"].notna().sum())

        station_id  latitude  longitude state
225    AQC00914000  -14.3167  -170.7667    AS
226    AQC00914005  -14.2667  -170.6500    AS
227    AQC00914021  -14.2667  -170.5833    AS
228    AQC00914060  -14.2667  -170.6833    AS
229    AQC00914135  -14.3000  -170.7000    AS
230    AQC00914138  -14.2833  -170.6833    AS
231    AQC00914141  -14.2667  -170.6167    AS
232    AQC00914145  -14.2833  -170.7167    AS
233    AQC00914149  -14.2833  -170.6833    AS
236    AQC00914397  -14.3500  -170.7833    AS
238    AQC00914594  -14.3333  -170.7667    AS
240    AQC00914822  -11.0500  -171.0833    AS
241    AQC00914869  -14.3333  -170.7167    AS
242    AQC00914873  -14.3500  -170.7667    AS
244    AQC00914912  -14.2500  -170.6667    AS
245    AQW00061705  -14.3306  -170.7136    AS
23870  CA001018611   48.0333  -123.3333    WA
24574  CA001102420   49.0000  -123.0833    WA
24595  CA001103635   49.0000  -122.2167    WA
24963  CA001135126   49.0000  -118.7667    WA
76034


In [17]:
#inner merge
# keep only needed columns from station table
df_state = (
    df_st_filled
    .dropna(subset=["state"])[["station_id", "state", "state_name"]]
)

# inner merge
df_merged = df_weekly.merge(
    df_state,
    on="station_id",
    how="inner"
)

print(df_merged.head())
df_merged.to_csv('climate_station_state_weekly.csv', index=False)

    station_id       date        PRCP  SNOW  SNWD  TMAX  TMIN state  \
0  AQC00914000 2021-01-03    0.000000   NaN   NaN   NaN   NaN    AS   
1  AQC00914000 2021-01-10  200.000000   NaN   NaN   NaN   NaN    AS   
2  AQC00914000 2021-01-17   63.571429   NaN   NaN   NaN   NaN    AS   
3  AQC00914000 2021-01-24   94.714286   NaN   NaN   NaN   NaN    AS   
4  AQC00914000 2021-01-31  464.857143   NaN   NaN   NaN   NaN    AS   

       state_name  
0  American Samoa  
1  American Samoa  
2  American Samoa  
3  American Samoa  
4  American Samoa  


In [19]:
## Aggregate by state
df_state_weekly = (
    df_merged
    .groupby(["state", "state_name","date"], as_index=False)
    .agg({
        "PRCP": "mean",
        "SNOW": "mean",
        "SNWD": "mean",
        "TMAX": "mean",
        "TMIN": "mean"
    })
)

print(df_state_weekly.head())
df_state_weekly.to_csv('climate_state_weekly.csv', index=False)

  state state_name       date       PRCP       SNOW        SNWD        TMAX  \
0    AK     Alaska 2021-01-03  15.757037   6.956229  443.230769 -120.400651   
1    AK     Alaska 2021-01-10  41.185132   9.730082  479.490637  -68.406274   
2    AK     Alaska 2021-01-17  49.561608   8.707667  506.491232  -47.916636   
3    AK     Alaska 2021-01-24  45.775236  10.574066  529.898891  -21.627442   
4    AK     Alaska 2021-01-31   7.832688   7.754924  550.206264 -115.577625   

         TMIN  
0 -184.843648  
1 -132.643198  
2 -107.757259  
3  -88.484158  
4 -190.838614  


## state lavel (monthly)
#### Documentation: https://www.ncei.noaa.gov/access/monitoring/climate-at-a-glance/statewide/time-series/service-api
#### Data: https://www.ncei.noaa.gov/pub/data/cirs/climdiv/

In [8]:
import pandas as pd
import requests
from io import StringIO
import xml.etree.ElementTree as ET

def load_noaa_state_data(state="41", variable="tavg", time_period="1",
                        start_year=2021, end_year=2026, fmt="csv"):
    
    url = f"https://www.ncei.noaa.gov/access/monitoring/climate-at-a-glance/statewide/time-series/{state}/{variable}/{time_period}/0/{start_year}-{end_year}/data.{fmt}"    
    
    r = requests.get(url)
    r.raise_for_status()
    return r

df = load_noaa_state_data()
print(df.text[:1000])

#  Texas Average Temperature
# Units: Degrees Fahrenheit
Date,Value
202101,47.3
202102,44
202103,59.4
202104,63.7
202105,71.9
202106,80.3
202107,80.9
202108,81.9
202109,78.2
202110,69.8
202111,57.3
202112,59.1
202201,45.8
202202,45.8
202203,57
202204,68.8
202205,78
202206,83.8
202207,87.2
202208,83.4
202209,77.8
202210,66.4
202211,54
202212,50
202301,51.6
202302,52.1
202303,60.4
202304,64.7
202305,73.8
202306,82.3
202307,86.3
202308,87.5
202309,81.8
202310,68.4
202311,56.5
202312,51.8
202401,44.3
202402,56.2
202403,60.4
202404,67.9
202405,76.9
202406,83.2
202407,82.8
202408,85.8
202409,77.5
202410,72.8
202411,60.8
202412,54.2
202501,42.8
202502,51.8
202503,63
202504,68.7
202505,74.2
202506,81.4
202507,82.2
202508,83.3
202509,78
202510,71.6
202511,62.1
202512,53.7
202601,47.4
202602,58.3

